# climate-toolkit — Colab quick start

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CGIAR-Climate-Data-Hub/climate-toolkit/blob/main/examples/climate_toolkit_colab.ipynb)

Location-based climate, season, climatology, hazard, and projection analysis, as a Python library.

This notebook accompanies the [Use as a package](https://CGIAR-Climate-Data-Hub.github.io/climate-toolkit/use-as-a-package/) guide and demos **all seven public functions**. It runs top-to-bottom on a fresh Colab runtime with **no credentials** — sections 1–4 use NASA POWER and weather stations, which need nothing. Section 5 (optional) unlocks the Earth Engine-backed sources (AgERA5, ERA5, CHIRPS, NEX-GDDP, ...) with a one-time free registration.

**Contents**
1. Install
2. Fetch daily climate data → pandas DataFrame
3. Seasonal climatology & statistics
4. Weather stations: download & validate a grid against observations
5. Optional — Earth Engine: gridded sources, hazards, comparisons, projections
6. Where to go next

## 1. Install

The package is not on PyPI yet, so install straight from GitHub. Takes ~1–2 minutes on Colab.

In [ ]:
%pip install -q "git+https://github.com/CGIAR-Climate-Data-Hub/climate-toolkit.git"

In [ ]:
import climate_toolkit as ct

print(f"climate_toolkit v{ct.__version__}")
print("Public API:", [n for n in ct.__all__ if not n.startswith("__")])

## 2. Fetch daily climate data → pandas DataFrame

`nasa_power` uses plain HTTPS — no credentials, no Earth Engine. We fetch one year of daily rainfall and temperature for Nairobi, Kenya. Swap in your own coordinates.

In [ ]:
from datetime import date

from climate_toolkit.fetch_data.source_data.sources.utils.models import ClimateVariable

LAT, LON = -1.286, 36.817  # Nairobi, Kenya — swap in your own site

VARS = [
    ClimateVariable.precipitation,
    ClimateVariable.max_temperature,
    ClimateVariable.min_temperature,
]

df = ct.fetch_climate_data(
    source="nasa_power",
    location_coord=(LAT, LON),
    variables=VARS,
    date_from=date(2020, 1, 1),
    date_to=date(2020, 12, 31),
    verbose=False,
)
print(f"{len(df)} daily rows")
df.head()

In [ ]:
# It's a normal DataFrame — plot, resample, export as usual.
df.set_index("date")["precipitation"].plot(
    figsize=(10, 3), title="Daily precipitation, Nairobi 2020 (NASA POWER)"
);

## 3. Seasonal climatology & statistics

`analyze_climate_statistics` detects growing seasons and returns a nested dict of per-season statistics, water balance (ET0, NDWS, WRSI), and long-term-mean summaries.

> A real climatology needs ~20+ years; the short window here keeps the demo fast and just prints a warning.

In [ ]:
stats = ct.analyze_climate_statistics(
    location_coord=(LAT, LON),
    start_year=2015,
    end_year=2020,
    source="nasa_power",
)
print("Result blocks:", sorted(stats.keys()))

In [ ]:
# Long-term-mean summary per detected season
stats["ltm_season_summary"]

## 4. Weather stations: download & validate

`download_station_data` finds the nearest GHCN-Daily (or GSOD) station and downloads its daily observations. No credentials needed. Arguments are keyword-only.

In [ ]:
station = ct.download_station_data(
    station_source="ghcn_daily",
    station_coord=(LAT, LON),
    date_from=date(2020, 1, 1),
    date_to=date(2020, 12, 31),
    max_distance_km=50.0,
    auto_select="auto-1",
)
station

### Validate a gridded product against the station

`compare_station_to_grids` checks how well a gridded dataset matches on-the-ground observations at your site — still credential-free with the `nasa_power` grid. (Add `"agera_5"` to `grid_sources` after the Earth Engine setup in section 5.)

> We restrict `variables` to precipitation here: the station nearest Nairobi records precipitation reliably but is too sparse on temperature to pass the completeness guard. Drop the `variables` argument at sites with fuller station records.

In [ ]:
validation = ct.compare_station_to_grids(
    station_source="ghcn_daily",
    station_coord=(LAT, LON),
    date_from=date(2019, 1, 1),
    date_to=date(2020, 12, 31),
    grid_sources=["nasa_power"],
    variables=[ClimateVariable.precipitation],
    verbose=False,
)
print("Validation blocks:", sorted(validation.keys()))
validation["confidence_summary"]

## 5. Optional — Earth Engine: gridded sources, hazards, comparisons, projections

Most gridded and projection sources (`agera_5`, `era_5`, `chirps_*`, `imerg`, `terraclimate`, `cmip_6`, `nex_gddp`, ...) route through **Google Earth Engine**. One-time setup, **free for noncommercial use** (research, academia, nonprofit):

1. Register a Cloud project at https://code.earthengine.google.com/register — choose **Unpaid usage** + a category like *Academia & Research*. No credit card needed.
2. Set `RUN_EARTH_ENGINE = True` and your project id below, then run the cell. Colab pops up a Google sign-in for `ee.Authenticate()`.

This section then walks through:

- **5.1** Fetch gridded daily data (AgERA5)
- **5.2** Seasonal statistics for a maize system
- **5.3** Crop hazard assessment
- **5.4** Focal year vs. baseline climatology
- **5.5** Side-by-side source comparison
- **5.6** Future climate projections (NEX-GDDP, 2050)

> **Runtimes.** Earth Engine cells pull multi-year daily data; expect a few minutes each, and ~10+ minutes for the 30-year baseline in 5.4. Results are cached under `outputs/cache/`, so re-runs are fast.

Full walkthrough: [Getting started → Google Earth Engine credentials](https://CGIAR-Climate-Data-Hub.github.io/climate-toolkit/getting_started/#2-google-earth-engine-credentials).

In [ ]:
RUN_EARTH_ENGINE = False  # set True after registering (free for noncommercial use)
GCP_PROJECT_ID = "your-ee-project-id"  # <-- your registered Cloud project id

# Tip: instead of pasting the id here, store it once in Colab's Secrets
# (key icon in the left sidebar, name it GCP_PROJECT_ID) and use:
#   from google.colab import userdata
#   GCP_PROJECT_ID = userdata.get("GCP_PROJECT_ID")

if RUN_EARTH_ENGINE:
    import os

    import ee

    ee.Authenticate()  # interactive Google sign-in (works natively in Colab)
    os.environ["GCP_PROJECT_ID"] = GCP_PROJECT_ID
    ee.Initialize(project=GCP_PROJECT_ID)
    print("Earth Engine ready")
else:
    print("Earth Engine section disabled — set RUN_EARTH_ENGINE = True above to run it.")

### 5.1 Fetch gridded daily data (AgERA5)

Same call as section 2 — only `source` changes. `agera_5` is the recommended default gridded source (temperature, humidity, wind, solar radiation + more).

In [ ]:
if RUN_EARTH_ENGINE:
    df_ee = ct.fetch_climate_data(
        source="agera_5",
        location_coord=(LAT, LON),
        variables=VARS,
        date_from=date(2020, 1, 1),
        date_to=date(2020, 12, 31),
        verbose=False,
    )
    display(df_ee.head())
else:
    print("Skipped — enable Earth Engine in 5 above.")

### 5.2 Seasonal statistics for a maize system

Same as section 3, but on the AgERA5 grid and with `crop_name` so season detection and water-balance parameters are crop-aware.

In [ ]:
if RUN_EARTH_ENGINE:
    stats_ee = ct.analyze_climate_statistics(
        location_coord=(LAT, LON),
        start_year=2015,
        end_year=2020,
        source="agera_5",
        crop_name="maize",
    )
    print("Result blocks:", sorted(stats_ee.keys()))
    display(stats_ee["ltm_season_summary"])
else:
    print("Skipped — enable Earth Engine in 5 above.")

### 5.3 Crop hazard assessment

`evaluate_hazards` screens a growing season for heat, drought, waterlogging and other crop/livestock hazards. `source="auto"` picks the recommended precipitation + temperature sources.

In [ ]:
if RUN_EARTH_ENGINE:
    hazards = ct.evaluate_hazards(
        crop_name="Maize",
        location_coord=(LAT, LON),
        date_from="2020-03-01",
        date_to="2020-08-31",
        source="auto",
    )
    print("Hazard blocks:", sorted(hazards.keys()))
else:
    print("Skipped — enable Earth Engine in 5 above.")

### 5.4 Focal year vs. baseline climatology

How did 2023 differ from the 1991–2020 baseline at this site? **Longest cell in the notebook** (~10+ min cold: it fetches 30 years of daily data). Cached afterwards.

In [ ]:
if RUN_EARTH_ENGINE:
    diff = ct.compare_climate_periods(
        location=(LAT, LON),
        baseline_start=1991,
        baseline_end=2020,
        focal_year=2023,
        source="agera_5",
        crop_name="maize",
    )
    print("Comparison blocks:", sorted(diff.keys()))
else:
    print("Skipped — enable Earth Engine in 5 above.")

### 5.5 Side-by-side source comparison

`compare_climate_sources` fetches the same site/period from multiple datasets and writes comparison reports to `output_dir`.

In [ ]:
if RUN_EARTH_ENGINE:
    import os

    comparison = ct.compare_climate_sources(
        sources=["nasa_power", "agera_5"],
        lat=LAT,
        lon=LON,
        start="2020-01-01",
        end="2020-12-31",
        output_dir="./outputs",
    )
    print("Wrote:", sorted(os.listdir("./outputs")))
else:
    print("Skipped — enable Earth Engine in 5 above.")

### 5.6 Future climate projections (NEX-GDDP, 2050)

Downscaled CMIP6 projections via `nex_gddp` — same fetch call plus a climate `model` and emissions `scenario`.

In [ ]:
if RUN_EARTH_ENGINE:
    proj = ct.fetch_climate_data(
        source="nex_gddp",
        model="GFDL-ESM4",
        scenario="ssp245",
        location_coord=(LAT, LON),
        variables=VARS,
        date_from=date(2050, 1, 1),
        date_to=date(2050, 12, 31),
        verbose=False,
    )
    display(proj.head())
    proj.set_index("date")["max_temperature"].plot(
        figsize=(10, 3), title="Projected daily max temperature, Nairobi 2050 (GFDL-ESM4, SSP2-4.5)"
    )
else:
    print("Skipped — enable Earth Engine in 5 above.")

## 6. Where to go next

You've now touched all seven public functions. To go deeper:

- **[Use as a package](https://CGIAR-Climate-Data-Hub.github.io/climate-toolkit/use-as-a-package/)** — the full guide: every function's parameters, data sources, variables, caching, recipes, troubleshooting.
- **[Getting started](https://CGIAR-Climate-Data-Hub.github.io/climate-toolkit/getting_started/)** — install and Earth Engine setup in detail.
- **[API reference](https://CGIAR-Climate-Data-Hub.github.io/climate-toolkit/api/)** — rendered from the docstrings; or run `help(ct.fetch_climate_data)` right here.

Issues and questions: https://github.com/CGIAR-Climate-Data-Hub/climate-toolkit/issues